In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Load data
data = pd.read_csv(r"C:\Users\aarna\train.csv")
data_test = pd.read_csv(r"C:\Users\aarna\test.csv")

# Convert Penalty column
def convert_penalty(p):
    if isinstance(p, str):
        if 's' in p:
            return int(p.replace('+', '').replace('s', ''))
        elif p == 'DNF':
            return 85
        elif p == 'DNS':
            return 95
    return np.nan

data['Penalty_numeric'] = data['Penalty'].apply(convert_penalty)
data_test['Penalty_numeric'] = data_test['Penalty'].apply(convert_penalty)
data.drop(columns=['Penalty'], inplace=True)
data_test.drop(columns=['Penalty'], inplace=True)

# Separate target
target = data['Lap_Time_Seconds']
data.drop(columns=['Lap_Time_Seconds'], inplace=True)

# Handle outliers and scale all at once
numeric_cols = data.select_dtypes(include=np.number).columns
for col in numeric_cols:
    Q1, Q3 = np.percentile(data[col], [25, 75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    data[col] = np.clip(data[col], lower, upper)
    data_test[col] = np.clip(data_test[col], lower, upper)

# Standard scale all numeric columns at once
scaler = StandardScaler()
data[numeric_cols] = scaler.fit_transform(data[numeric_cols])
data_test[numeric_cols] = scaler.transform(data_test[numeric_cols])

# Handle categorical variables
top_n = 30
top_categories = {}
cat_cols = data.select_dtypes(exclude=np.number).columns.tolist()

for col in cat_cols:
    top_values = data[col].value_counts().nlargest(top_n).index
    data[col] = data[col].where(data[col].isin(top_values), other='Other')
    data_test[col] = data_test[col].where(data_test[col].isin(top_values), other='Other')
    top_categories[col] = top_values

# One-hot encoding
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore').set_output(transform='pandas')
ohe.fit(data[cat_cols])
data_encoded = ohe.transform(data[cat_cols])
data_test_encoded = ohe.transform(data_test[cat_cols])

# Final combined data
data = data.drop(columns=cat_cols).join(data_encoded)
data_test = data_test.drop(columns=cat_cols).join(data_test_encoded)

# Match test columns
data_test = data_test.reindex(columns=data.columns, fill_value=0)

# Split train/val
X_train, X_val, y_train, y_val = train_test_split(data, target, test_size=0.2, random_state=42)

# Train random forest with optimization
model = RandomForestRegressor(n_estimators=50, max_depth=12, n_jobs=-1, random_state=42)
model.fit(data, target)
y_pred = model.predict(data_test)
df_predictions = pd.DataFrame({
    'Unique ID': data_test['Unique ID'].astype('int'),  # or whatever your ID column is
    'Lap_Time_Seconds': y_pred
})



# Evaluate
#rmse = mean_squared_error(y_val, y_pred, squared=False)
#print("RMSE:", rmse)


SAVING THE MODEL


In [ ]:
#tree_model.fit(x_train, y_train)
#val_preds = tree_model.predict(x_test)

import joblib

# Save model
joblib.dump(tree_model, 'tree_model.pkl')
print("hi")

